# Simbay MJX 100-Particle Demo

This notebook is designed for Google Colab. It clones the current repository from GitHub, installs the required runtime dependencies, imports the project modules directly from the cloned checkout, and runs a simple headless MJX particle-filter demo with 100 particles.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/marianscosta/simbay.git"
REPO_DIR = Path("/content/simbay")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "mujoco==3.6.0",
        "mujoco-mjx==3.6.0",
        "jax==0.5.0",
        "jaxlib==0.5.0",
        "matplotlib",
    ],
    check=True,
)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f"Cloned repo into: {REPO_DIR}")
print(f"Working directory: {Path.cwd()}")


In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from src.estimation import FrankaMJXEnv
from src.estimation import ParticleFilter
from src.utils import DEFAULT_OBJECT_PROPS
from src.utils import FRANKA_HOME_QPOS
from src.utils import initialize_mujoco_env

np.random.seed(0)

NUM_PARTICLES = 100
MASS_LIMITS = (0.0, 3.0)
SENSOR_NOISE_STD = 0.5
STEPS_PER_SEGMENT = 40

OPEN_GRIPPER = 255
Q_HOME = np.append(FRANKA_HOME_QPOS, OPEN_GRIPPER)

print(f"True object mass: {DEFAULT_OBJECT_PROPS['mass']:.3f} kg")
print(f"Home configuration shape: {Q_HOME.shape}")


In [ ]:
def interpolate_waypoints(waypoints, steps_per_segment):
    controls = []
    for start, end in zip(waypoints[:-1], waypoints[1:]):
        segment = np.linspace(start, end, steps_per_segment + 1)[1:]
        controls.extend(segment)
    return np.asarray(controls)


waypoints = [
    Q_HOME,
    Q_HOME + np.array([0.00, 0.10, 0.00, 0.00, 0.00, 0.00, 0.00, 0.0]),
    Q_HOME + np.array([0.05, 0.10, -0.08, 0.10, 0.00, 0.00, 0.00, 0.0]),
    Q_HOME + np.array([0.05, -0.05, -0.08, 0.05, 0.05, 0.00, 0.00, 0.0]),
    Q_HOME,
]

control_sequence = interpolate_waypoints(waypoints, STEPS_PER_SEGMENT)
print(f"Number of control steps: {len(control_sequence)}")
print(f"First control vector: {control_sequence[0]}")


In [ ]:
real_robot = initialize_mujoco_env()
env = FrankaMJXEnv(MASS_LIMITS, NUM_PARTICLES)
particle_filter = ParticleFilter(env)

profile = env.memory_profile()
print("MJX execution profile:")
for key, value in profile.items():
    print(f"  {key}: {value}")

print(f"Initial estimate: {float(particle_filter.estimate()):.4f} kg")


In [ ]:
estimate_history = []
ess_history = []
step_wall_ms = []

for control in control_sequence:
    real_robot.move_joints(control)

    wall_start = time.perf_counter()
    particle_filter.predict(control)

    measurement = real_robot.get_sensor_reads()
    noisy_measurement = measurement + np.random.normal(0.0, SENSOR_NOISE_STD, size=3)

    particle_filter.update(noisy_measurement)
    particle_filter.resample()
    step_wall_ms.append((time.perf_counter() - wall_start) * 1000.0)

    estimate_history.append(float(particle_filter.estimate()))
    ess_history.append(float(particle_filter.effective_sample_size()))

final_estimate = estimate_history[-1]
true_mass = float(DEFAULT_OBJECT_PROPS["mass"])
error_pct = abs(true_mass - final_estimate) / true_mass * 100.0

print(f"Final estimate: {final_estimate:.4f} kg")
print(f"True mass: {true_mass:.4f} kg")
print(f"Final error: {error_pct:.2f}%")
print(f"Average step time: {np.mean(step_wall_ms):.2f} ms")
print(f"Final ESS: {ess_history[-1]:.2f}")


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

axes[0].plot(estimate_history, color="tab:red", linewidth=2, label="Estimate")
axes[0].axhline(true_mass, color="tab:green", linestyle="--", linewidth=2, label="True mass")
axes[0].set_ylabel("Mass (kg)")
axes[0].set_title("MJX Particle Filter Estimate")
axes[0].legend()
axes[0].grid(True, linestyle=":", alpha=0.7)

axes[1].plot(ess_history, color="tab:blue", linewidth=2)
axes[1].set_ylabel("ESS")
axes[1].set_xlabel("Step")
axes[1].set_title("Effective Sample Size")
axes[1].grid(True, linestyle=":", alpha=0.7)

plt.tight_layout()
plt.show()
